In [ ]:
import sys
sys.path.append('..') # Allows notebook to find 'src'

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests
import json

# Import the new orchestrator and execution functions
from src.event_core import (
    orchestrate_request, 
    execute_confirmed_task, 
    load_state, 
    process_long_document
)

In [ ]:
# --- UTILITIES ---
def get_available_models():
    try:
        response = requests.get('http://localhost:11434/api/tags')
        response.raise_for_status()
        models = response.json().get('models', [])
        ollama_models = [model['name'] for model in models] if models else []
    except requests.exceptions.RequestException:
        ollama_models = []
    together_models = [
        'togetherai/meta-llama/Llama-3-8b-chat-hf',
        'togetherai/mistralai/Mixtral-8x7B-Instruct-v0.1',
        'togetherai/Qwen/Qwen1.5-7B-Chat'
    ]
    return ollama_models + together_models

# --- GLOBAL STATE --- 
conversation_state = {}
task_queue = []
current_task_index = 0
total_tasks = 0

# --- UI WIDGETS ---
header = widgets.HTML(
    "<h1>Event Management Assistant 🎯</h1>"
    "<p>I'll help you manage your event venues and sessions. "
    "You can ask single questions or paste documents with multiple tasks.</p>"
)

available_models = get_available_models()
model_selector = widgets.Dropdown(
    options=available_models, 
    value=(available_models[0] if available_models else None), 
    description='Model:', 
    style={'description_width': 'initial'}
)

role_selector = widgets.RadioButtons(
    options=['admin', 'scheduler'], 
    value='admin', 
    description='Role:'
)

# State display
state_html_view = widgets.HTML()
state_accordion = widgets.Accordion(
    children=[state_html_view], 
    titles=('📊 Current Conference State',)
)
state_accordion.selected_index = None  # Start collapsed

# Progress indicator for multi-task processing
progress_label = widgets.HTML()

# Chat interface
chat_history_box = widgets.VBox([])
user_input = widgets.Textarea(
    placeholder='Type your request or paste your document here...', 
    layout={'width': '95%', 'height': '100px'}
)

# Buttons
send_button = widgets.Button(
    description="Send", 
    button_style='success', 
    icon='paper-plane'
)
clear_button = widgets.Button(
    description="Clear Chat", 
    icon='trash', 
    button_style='warning'
)
confirm_button = widgets.Button(
    description="✅ Confirm & Execute", 
    button_style='success', 
    layout={'visibility': 'hidden'}
)
cancel_button = widgets.Button(
    description="❌ Cancel", 
    button_style='danger', 
    layout={'visibility': 'hidden'}
)

# --- UI HELPER FUNCTIONS ---
def add_message_to_chat(message_html, is_user=False, is_system=False):
    if is_user:
        bubble_class = 'user-bubble'
    elif is_system:
        bubble_class = 'system-bubble'
    else:
        bubble_class = 'assistant-bubble'
    
    full_html = f"<div class='chat-bubble {bubble_class}'>{message_html}</div>"
    chat_history_box.children = list(chat_history_box.children) + [widgets.HTML(full_html)]

def show_confirmation_buttons(show=True):
    visibility = 'visible' if show else 'hidden'
    confirm_button.layout.visibility = visibility
    cancel_button.layout.visibility = visibility
    user_input.disabled = show
    send_button.disabled = show

def update_progress():
    global current_task_index, total_tasks
    if total_tasks > 1:
        progress_label.value = (
            f"<div style='background-color: #e3f2fd; padding: 8px; "
            f"border-radius: 5px; margin: 5px 0;'>"
            f"📋 Processing task {current_task_index} of {total_tasks}"
            f"</div>"
        )
    else:
        progress_label.value = ""

# --- STATE DISPLAY ---
def update_state_display():
    try:
        current_state = load_state()
        venues = current_state.get('venues', {})
        sessions = current_state.get('sessions', [])
        bookings = current_state.get('venue_bookings', {})
        
        # Venues table
        venues_html = (
            "<h4>Venues</h4>"
            "<table border='1' style='width:100%; border-collapse: collapse;'>"
            "<tr style='background-color: #f0f0f0;'>"
            "<th style='padding:8px;'>Name</th>"
            "<th style='padding:8px;'>Capacity</th>"
            "<th style='padding:8px;'>A/V System</th>"
            "<th style='padding:8px;'>Status</th>"
            "</tr>"
        )
        
        if not venues:
            venues_html += (
                "<tr><td colspan='4' style='padding:8px; text-align:center;'>"
                "<i>No venues created yet</i></td></tr>"
            )
        else:
            for name, props in sorted(venues.items()):
                status = "Booked" if name in bookings else "Available"
                status_color = "#ffcdd2" if status == "Booked" else "#c8e6c9"
                av_icon = "✅" if props.get('has_av_system') else "❌"
                venues_html += (
                    f"<tr>"
                    f"<td style='padding:8px;'>{name}</td>"
                    f"<td style='padding:8px; text-align:center;'>{int(props.get('capacity', 0))}</td>"
                    f"<td style='padding:8px; text-align:center;'>{av_icon}</td>"
                    f"<td style='padding:8px; background-color:{status_color}; text-align:center;'>{status}</td>"
                    f"</tr>"
                )
        venues_html += "</table>"
        
        # Sessions table
        sessions_html = (
            "<h4 style='margin-top: 20px;'>Scheduled Sessions</h4>"
            "<table border='1' style='width:100%; border-collapse: collapse;'>"
            "<tr style='background-color: #f0f0f0;'>"
            "<th style='padding:8px;'>Session Name</th>"
            "<th style='padding:8px;'>Venue</th>"
            "<th style='padding:8px;'>Host</th>"
            "<th style='padding:8px;'>Attendees</th>"
            "<th style='padding:8px;'>A/V Required</th>"
            "</tr>"
        )
        
        if not sessions:
            sessions_html += (
                "<tr><td colspan='5' style='padding:8px; text-align:center;'>"
                "<i>No sessions scheduled yet</i></td></tr>"
            )
        else:
            for session in sessions:
                av_icon = "✅" if session.get('requires_av') else "❌"
                sessions_html += (
                    f"<tr>"
                    f"<td style='padding:8px;'>{session.get('name', 'N/A')}</td>"
                    f"<td style='padding:8px;'>{session.get('in_venue', 'N/A')}</td>"
                    f"<td style='padding:8px;'>{session.get('hosted_by', 'N/A')}</td>"
                    f"<td style='padding:8px; text-align:center;'>{session.get('expected_attendees', 'N/A')}</td>"
                    f"<td style='padding:8px; text-align:center;'>{av_icon}</td>"
                    f"</tr>"
                )
        sessions_html += "</table>"
        
        state_html_view.value = venues_html + sessions_html
        
    except Exception as e:
        state_html_view.value = f"<p style='color:red;'>Error loading state: {e}</p>"

# --- CORE CHAT LOGIC ---
def process_next_task_in_queue():
    global conversation_state, task_queue, current_task_index, total_tasks
    
    if not task_queue:
        if total_tasks > 1:
            add_message_to_chat(
                f"<b>✅ Completed all {total_tasks} tasks!</b>",
                is_system=True
            )
        progress_label.value = ""
        total_tasks = 0
        current_task_index = 0
        return
    
    current_task_index = total_tasks - len(task_queue) + 1
    update_progress()
    
    query = task_queue.pop(0)
    add_message_to_chat(
        f"<b>Task {current_task_index}:</b> {query}",
        is_system=True
    )
    handle_user_query(query, is_from_queue=True)

def handle_user_query(query, is_from_queue=False):
    global conversation_state
    
    if not is_from_queue:
        add_message_to_chat(f"<b>You:</b> {query}", is_user=True)
        user_input.value = ''
    
    send_button.disabled = True
    
    # Add thinking indicator
    thinking_msg = widgets.HTML(
        "<div class='chat-bubble assistant-bubble'>"
        "<i>Processing your request...</i></div>"
    )
    chat_history_box.children = list(chat_history_box.children) + [thinking_msg]
    
    try:
        # Orchestrate the request
        result = orchestrate_request(
            query, 
            role_selector.value, 
            model_selector.value, 
            conversation_state
        )
        
        conversation_state = result.get('new_state', conversation_state)
        
        # Remove thinking message
        chat_history_box.children = chat_history_box.children[:-1]
        
        if result['status'] == 'clarification_needed':
            # Show understanding
            if 'understanding_html' in result:
                add_message_to_chat(result['understanding_html'])
            # Ask for clarification
            add_message_to_chat(
                f"<b>Assistant:</b><br>{result['message']}"
            )
        
        elif result['status'] == 'confirmation_needed':
            # Show final understanding
            if 'understanding_html' in result:
                add_message_to_chat(result['understanding_html'])
            # Ask for confirmation
            add_message_to_chat(result['message'])
            show_confirmation_buttons(True)
        
        else:  # Error case
            add_message_to_chat(
                f"<b>Assistant:</b><br>"
                f"<span style='color:red;'>{result['message']}</span>"
            )
            reset_conversation()
            process_next_task_in_queue()
    
    except Exception as e:
        # Remove thinking message if it exists
        if chat_history_box.children and 'Processing' in str(chat_history_box.children[-1].value):
            chat_history_box.children = chat_history_box.children[:-1]
        
        add_message_to_chat(
            f"<b>Assistant:</b><br>"
            f"<span style='color:red;'>An error occurred: {e}</span>"
        )
        reset_conversation()
    
    send_button.disabled = False

# --- BUTTON CLICK HANDLERS ---
def on_send_button_clicked(b):
    global conversation_state, task_queue, total_tasks
    
    query = user_input.value.strip()
    if not query:
        return
    
    # Check if this is a document (long text with multiple potential tasks)
    if len(query) > 300 and not conversation_state:
        add_message_to_chat(
            f"<b>You:</b> [Submitted a document with {len(query)} characters]",
            is_user=True
        )
        user_input.value = ''
        
        # Process document
        add_message_to_chat(
            "<i>Analyzing document for tasks...</i>"
        )
        
        result = process_long_document(
            query,
            role_selector.value,
            model_selector.value
        )
        
        # Remove analyzing message
        chat_history_box.children = chat_history_box.children[:-1]
        
        add_message_to_chat(
            f"<b>Assistant:</b> {result['message']}"
        )
        
        if result['status'] == 'tasks_extracted' and result['tasks']:
            task_queue = result['tasks']
            total_tasks = len(task_queue)
            process_next_task_in_queue()
    else:
        # Single query
        handle_user_query(query)

def on_confirm_button_clicked(b):
    global conversation_state, task_queue
    
    show_confirmation_buttons(False)
    
    # Add executing message
    add_message_to_chat("<i>Executing task...</i>")
    
    # Execute the task
    result = execute_confirmed_task(
        role_selector.value,
        conversation_state
    )
    
    # Remove executing message
    chat_history_box.children = chat_history_box.children[:-1]
    
    if result['status'] == 'success':
        # Success message
        add_message_to_chat(
            f"<b>Assistant:</b> {result['message']}",
            is_system=True
        )
        
        # Show DSL in collapsible accordion
        if 'dsl_code' in result and result['dsl_code']:
            dsl_html = (
                f"<pre style='background-color:#f5f5f5; "
                f"border: 1px solid #ddd; padding: 10px; "
                f"border-radius: 5px; font-size: 12px;'>"
                f"{result['dsl_code']}</pre>"
            )
            dsl_accordion = widgets.Accordion(
                children=[widgets.HTML(dsl_html)],
                titles=('📝 Generated DSL Code',)
            )
            dsl_accordion.selected_index = None  # Start collapsed
            chat_history_box.children = list(chat_history_box.children) + [dsl_accordion]
        
        # Update state display
        update_state_display()
    else:
        # Error message
        add_message_to_chat(
            f"<b>Assistant:</b><br>"
            f"<span style='color:red;'>{result['message']}</span>"
        )
    
    reset_conversation()
    process_next_task_in_queue()

def on_cancel_button_clicked(b):
    global conversation_state
    
    show_confirmation_buttons(False)
    add_message_to_chat(
        "<b>Assistant:</b> Task cancelled.",
        is_system=True
    )
    reset_conversation()
    process_next_task_in_queue()

def reset_conversation():
    global conversation_state
    conversation_state = {}
    user_input.disabled = False
    send_button.disabled = False

def on_clear_button_clicked(b):
    global task_queue, conversation_state, total_tasks, current_task_index
    
    reset_conversation()
    task_queue = []
    total_tasks = 0
    current_task_index = 0
    progress_label.value = ""
    
    initial_msg = widgets.HTML(
        "<div class='chat-bubble assistant-bubble'>"
        "<b>Assistant:</b><br>"
        "Hello! I'm ready to help you manage your event venues and sessions. "
        "You can:<br>"
        "• Create or modify venues<br>"
        "• Schedule sessions<br>"
        "• Paste documents with multiple tasks<br><br>"
        "What would you like to do?"
        "</div>"
    )
    chat_history_box.children = [initial_msg]
    show_confirmation_buttons(False)
    update_state_display()

# --- WIRE UP EVENT HANDLERS ---
send_button.on_click(on_send_button_clicked)
clear_button.on_click(on_clear_button_clicked)
confirm_button.on_click(on_confirm_button_clicked)
cancel_button.on_click(on_cancel_button_clicked)

# --- STYLES ---
styles = HTML("""
<style>
.chat-bubble {
    max-width: 80%;
    padding: 12px;
    border-radius: 10px;
    margin: 8px 0;
    line-height: 1.5;
}
.user-bubble {
    background-color: #E3F2FD;
    margin-left: 20%;
    border: 1px solid #90CAF9;
}
.assistant-bubble {
    background-color: #F3E5F5;
    margin-right: 20%;
    border: 1px solid #CE93D8;
}
.system-bubble {
    background-color: #E8F5E9;
    margin-left: auto;
    margin-right: auto;
    width: 60%;
    text-align: center;
    border: 1px solid #A5D6A7;
}
code {
    background-color: #f5f5f5;
    padding: 2px 4px;
    border-radius: 3px;
    font-family: monospace;
}
</style>
""")

display(styles)

# --- INITIAL SETUP ---
on_clear_button_clicked(None)

# --- BUILD UI LAYOUT ---
input_box = widgets.HBox([user_input, send_button, clear_button])
confirmation_box = widgets.HBox([confirm_button, cancel_button])

# Main layout
main_layout = widgets.VBox([
    header,
    widgets.HBox([model_selector, role_selector]),
    widgets.HTML("<hr>"),
    state_accordion,
    progress_label,
    chat_history_box,
    input_box,
    confirmation_box
])

display(main_layout)